# NN_14 — 1D CNN com 4 Features como Sequência

**Questão**: uma arquitetura CNN traz vantagem mesmo quando o input são apenas
as 4 features extraídas manualmente — em vez do sinal bruto z_eq?

As 4 features são tratadas como uma *sequência de comprimento 4*: `Input(4, 1)`.
A CNN aprende filtros que combinam as features de forma não-linear, com
compartilhamento de pesos entre posições (inductive bias diferente do DNN).

| Modelo | Input shape | Params |
|--------|-------------|--------|
| DNN 4-feat (NN_02b) | (4,) | ~44k |
| **CNN 4-feat (este)** | (4, 1) | ~1k |
| 1D CNN z_eq (NN_02 GPU) | (1024, 1) | ~69k |

**Nota**: com comprimento 4, os filtros Conv1D são limitados a k≤4.
A arquitetura é deliberadamente pequena — a questão é o inductive bias, não a capacidade.

Modelo salvo em `model_cnn_4feat.keras`.
Features do GAP salvas para uso no NN_11 (CNN-SVM híbrido).

In [ ]:
# ==============================================================================
# 1. IMPORTS & GPU
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
import h5py
import json
from pathlib import Path
from scipy.stats import norm
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy

print(f'TensorFlow : {tf.__version__}')

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus: tf.config.experimental.set_memory_growth(g, True)
    USE_GPU = True
    keras.mixed_precision.set_global_policy('mixed_float16')
    print(f'GPU: {len(gpus)}x  |  mixed_float16')
else:
    USE_GPU = False
    print('No GPU — CPU mode')

tf.config.optimizer.set_jit(True)
np.random.seed(42)
tf.random.set_seed(42)

project_root       = Path.cwd().parent
results_dir        = project_root / 'results'
data_dir           = results_dir / 'data'
models_dir         = results_dir / 'models'
vis_dir            = results_dir / 'visualizations'
models_dir.mkdir(parents=True, exist_ok=True)
vis_dir.mkdir(parents=True, exist_ok=True)

ALPHA    = 1e-7
Q_INV    = norm.ppf(1 - ALPHA)
SNR_BINS = list(range(0, 31, 5))
print(f'α={ALPHA:.0e}  Q⁻¹={Q_INV:.4f}')

In [ ]:
# ==============================================================================
# 2. CARREGAR DATASET E EXTRAIR 4 FEATURES
# ==============================================================================
# Mesmas 4 features do NN_02b: [|τ|, ĥ, SNR, E]
# Aqui são tratadas como sequência (4, 1) para a Conv1D.

dataset_path = data_dir / 'dataset_cnn_yeq_0_30dB.h5'

with h5py.File(str(dataset_path), 'r') as f:
    L_FIXED = int(f.attrs['L_FIXED'])
    RHO_T   = float(f.attrs['RHO_T'])
    RHO_S   = float(f.attrs['RHO_S'])

    Y_train   = f['train/y_eq'][:].astype(np.float32)
    TAU_train = np.abs(f['train/tau_eq'][:]).astype(np.float32)
    snr_train = f['train/snr'][:]
    y_train   = f['train/y'][:].astype(np.float32)

    Y_val     = f['val/y_eq'][:].astype(np.float32)
    TAU_val   = np.abs(f['val/tau_eq'][:]).astype(np.float32)
    snr_val   = f['val/snr'][:]
    y_val     = f['val/y'][:].astype(np.float32)

    Y_test    = f['test/y_eq'][:].astype(np.float32)
    TAU_test  = np.abs(f['test/tau_eq'][:]).astype(np.float32)
    snr_test  = f['test/snr'][:]
    y_test    = f['test/y'][:].astype(np.float32)

def extract_features(Y, TAU, snr):
    h_est = np.abs(Y).mean(axis=1).astype(np.float32)
    E     = (Y ** 2).mean(axis=1).astype(np.float32)
    return np.stack([TAU, h_est, snr.astype(np.float32), E], axis=1)

F_train = extract_features(Y_train, TAU_train, snr_train)  # (N, 4)
F_val   = extract_features(Y_val,   TAU_val,   snr_val)
F_test  = extract_features(Y_test,  TAU_test,  snr_test)

# Normalização z-score por feature (idêntica ao NN_02b)
sc_mean = F_train.mean(axis=0).astype(np.float64)
sc_std  = F_train.std(axis=0).astype(np.float64)

F_train_n = ((F_train - sc_mean) / sc_std).astype(np.float32)
F_val_n   = ((F_val   - sc_mean) / sc_std).astype(np.float32)
F_test_n  = ((F_test  - sc_mean) / sc_std).astype(np.float32)

# Reshape para (N, 4, 1) — entrada Conv1D
X_train = F_train_n[:, :, np.newaxis]   # (N, 4, 1)
X_val   = F_val_n[:,   :, np.newaxis]
X_test  = F_test_n[:,  :, np.newaxis]

print(f'Features shape  : {X_train.shape}  (N, 4, 1)')
print(f'Train: H0={(y_train==0).sum()}  H1={(y_train==1).sum()}')
print(f'Val  : {X_val.shape}')
print(f'\nFeatures normalizadas (treino):  mean≈{F_train_n.mean(0).round(2)}  std≈{F_train_n.std(0).round(2)}')

In [ ]:
# ==============================================================================
# 3. ARQUITETURA — CNN para 4 features (Input shape = 4 × 1)
# ==============================================================================
# Sequência de comprimento 4 → filtros conv de tamanho 2 ou 3.
# GAP reduz (4, n_filters) → (n_filters,) — representação usada pelo SVM no NN_11.
#
# Bloco 1: Conv1D(16, k=2) → BN → ReLU   [→ (4, 16) com padding='same']
# Bloco 2: Conv1D(32, k=2) → BN → ReLU   [→ (4, 32)]
# GAP: GlobalAvgPool       [→ (32,)]      ← features para SVM
# Head: Dense(16, relu) → Dense(1, sigmoid)
#
# Total: ~1k params vs ~44k do DNN 4-feat.
# A questão não é capacidade — é se o inductive bias conv ajuda em 4 posições.

def build_cnn_4feat():
    inp = layers.Input(shape=(4, 1), name='features_input')

    x = layers.Conv1D(16, kernel_size=2, padding='same',
                      use_bias=False, kernel_initializer='he_uniform')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv1D(32, kernel_size=2, padding='same',
                      use_bias=False, kernel_initializer='he_uniform')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    gap = layers.GlobalAveragePooling1D(name='gap_features')(x)   # (32,) — para SVM

    x = layers.Dense(16, activation='relu',
                     kernel_initializer='he_uniform')(gap)
    x = layers.Dropout(0.2)(x)

    out = layers.Dense(1, activation='sigmoid', dtype='float32', name='P_H1')(x)
    return keras.Model(inputs=inp, outputs=out, name='CNN_4feat')


model = build_cnn_4feat()
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=BinaryCrossentropy(),
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)
model.summary()
print(f'\nTotal params : {model.count_params():,}')

In [ ]:
# ==============================================================================
# 4. CALLBACKS & TREINAMENTO
# ==============================================================================

model_path = str(models_dir / 'model_cnn_4feat.keras')

cbs = [
    callbacks.EarlyStopping(
        monitor='val_auc', mode='max', patience=20,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ModelCheckpoint(
        model_path, monitor='val_auc', mode='max',
        save_best_only=True, verbose=0
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=8,
        min_lr=1e-6, verbose=1
    ),
]

BATCH_SIZE = 512

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=BATCH_SIZE,
    callbacks=cbs,
    verbose=1
)

print(f'\nBest val AUC  : {max(history.history["val_auc"]):.5f}')
print(f'Best val loss : {min(history.history["val_loss"]):.5f}')
print(f'Epochs run    : {len(history.history["loss"])}')

In [ ]:
# ==============================================================================
# 5. D3F THRESHOLD & PD vs SNR
# ==============================================================================

p_val  = model.predict(X_val,  batch_size=1024, verbose=0).flatten()
p_test = model.predict(X_test, batch_size=1024, verbose=0).flatten()

auc_val  = roc_auc_score(y_val,  p_val)
auc_test = roc_auc_score(y_test, p_test)
print(f'Val  AUC : {auc_val:.5f}')
print(f'Test AUC : {auc_test:.5f}')

SNR_POINTS = np.array(SNR_BINS, dtype=float)
HALF       = 2.5

thresholds = {}
pd_cnn4feat = {}

print(f'\nD3F calibration (α={ALPHA:.0e}):')  
print(f'{"SNR":>5}  {"N_H0":>7}  {"μ_H0":>9}  {"σ_H0":>9}  {"τ*":>10}  {"PD":>8}')
print('-' * 55)

for snr_db in SNR_BINS:
    m0_v = (y_val  == 0) & (snr_val  >= snr_db-HALF) & (snr_val  < snr_db+HALF)
    m1_t = (y_test == 1) & (snr_test >= snr_db-HALF) & (snr_test < snr_db+HALF)

    if m0_v.sum() < 30 or m1_t.sum() < 5:
        thresholds[snr_db]   = np.nan
        pd_cnn4feat[snr_db] = np.nan
        continue

    s_h0 = p_val[m0_v]
    mu, sig = float(s_h0.mean()), float(s_h0.std())
    thr = float(np.clip(mu + Q_INV * sig, None, 1.0))
    pd  = float((p_test[m1_t] >= thr).mean())

    thresholds[snr_db]   = thr
    pd_cnn4feat[snr_db] = pd
    print(f'{snr_db:>5}  {m0_v.sum():>7}  {mu:>9.5f}  {sig:>9.5f}  {thr:>10.6f}  {pd:>8.4f}')

In [ ]:
# ==============================================================================
# 6. COMPARAÇÃO: CNN-4feat vs DNN-4feat vs 1D CNN (se disponível)
# ==============================================================================

# Carregar DNN-4feat (NN_02b)
dnn4_path = models_dir / 'model_dnn_4feat_aligned.keras'
pd_dnn4feat = {}
if dnn4_path.exists():
    from sklearn.metrics import roc_auc_score as _auc
    dnn4 = keras.models.load_model(str(dnn4_path))
    scaler_path = models_dir / 'scaler_4feat_aligned.json'
    sc = json.load(open(str(scaler_path)))
    sc_mean_d, sc_std_d = np.array(sc['mean']), np.array(sc['std'])
    # DNN usa features flat (N, 4), não (N, 4, 1)
    p_dnn_test = dnn4.predict(
        ((F_test - sc_mean_d) / sc_std_d).astype(np.float32),
        batch_size=1024, verbose=0
    ).flatten()
    # Re-calibrate with same D3F approach using val scores from DNN
    p_dnn_val = dnn4.predict(
        ((F_val  - sc_mean_d) / sc_std_d).astype(np.float32),
        batch_size=1024, verbose=0
    ).flatten()
    for snr_db in SNR_BINS:
        m0_v = (y_val == 0) & (snr_val >= snr_db-HALF) & (snr_val < snr_db+HALF)
        m1_t = (y_test == 1) & (snr_test >= snr_db-HALF) & (snr_test < snr_db+HALF)
        if m0_v.sum() < 30 or m1_t.sum() < 5:
            pd_dnn4feat[snr_db] = np.nan; continue
        mu, sig = p_dnn_val[m0_v].mean(), p_dnn_val[m0_v].std()
        thr = float(np.clip(mu + Q_INV * sig, None, 1.0))
        pd_dnn4feat[snr_db] = float((p_dnn_test[m1_t] >= thr).mean())
    print('DNN 4-feat carregado e avaliado com D3F.')
else:
    print('DNN 4-feat não encontrado — execute NN_02b primeiro.')

# Carregar 1D CNN results (NN_07)
fig2_path = data_dir / 'figure2_pd_vs_snr_gpu.json'
pd_cnn1d = {}
if fig2_path.exists():
    fig2 = json.load(open(str(fig2_path)))
    pd_cnn1d = {int(k): (v if v is not None else np.nan)
                for k, v in fig2['method_cnn']['PD_vs_SNR'].items()}
    pd_classical = {int(k): (v if v is not None else np.nan)
                    for k, v in fig2['method_tau']['PD_vs_SNR'].items()}
    print('1D CNN e Classical carregados do NN_07.')
else:
    pd_classical = {}
    print('NN_07 não encontrado — execute após NN_02 GPU.')

# Gráfico
snr_v = np.array(SNR_BINS, dtype=float)
get   = lambda d: np.array([d.get(s, np.nan) for s in SNR_BINS])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'CNN 4-feat vs DNN 4-feat vs 1D CNN z_eq  (α={ALPHA:.0e})',
             fontsize=12, fontweight='bold')

ax = axes[0]
if pd_classical:
    ax.plot(snr_v, get(pd_classical), 'o-',  color='tomato',     lw=2.5, ms=7, label='Classical')
if pd_dnn4feat:
    ax.plot(snr_v, get(pd_dnn4feat),  '^--', color='darkorange',  lw=2,   ms=7, label='DNN 4-feat (NN_02b)')
ax.plot(snr_v, get(pd_cnn4feat),      's-.', color='mediumpurple',lw=2,   ms=7, label='CNN 4-feat (este)')
if pd_cnn1d:
    ax.plot(snr_v, get(pd_cnn1d),     'D:',  color='steelblue',  lw=2,   ms=7, label='1D CNN z_eq (NN_02 GPU)')
ax.set(title='PD vs SNR', xlabel='SNR (dB)', ylabel='PD',
       xlim=(-1,31), ylim=(-0.05,1.05))
ax.set_xticks(SNR_BINS); ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(history.history['loss'],     label='Train Loss', lw=2, color='steelblue')
ax2.plot(history.history['val_loss'], label='Val Loss',   lw=2, color='steelblue', ls='--')
ax2b = ax2.twinx()
ax2b.plot(history.history['auc'],     label='Train AUC', lw=2, color='tomato')
ax2b.plot(history.history['val_auc'], label='Val AUC',   lw=2, color='tomato', ls='--')
ax2b.set_ylabel('AUC', color='tomato'); ax2b.set_ylim(0.4, 1.02)
ax2.set(title='Curva de Aprendizado — CNN 4-feat', xlabel='Época', ylabel='Loss')
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1+lines2, labels1+labels2, fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
fig_path = vis_dir / 'NN14_CNN_4feat_comparison.png'
plt.savefig(str(fig_path), dpi=150)
plt.show()
print(f'Figura salva → {fig_path}')

In [ ]:
# ==============================================================================
# 7. SALVAR FEATURES GAP PARA NN_11 (CNN-SVM híbrido)
# ==============================================================================
# O NN_11 usa as features do GlobalAveragePooling da CNN como input do SVM.
# Aqui extrai-se o vetor GAP (32-dim) para cada split e salva em HDF5.

gap_extractor = keras.Model(
    inputs=model.input,
    outputs=model.get_layer('gap_features').output,
    name='CNN4feat_GAP'
)

GAP_train = gap_extractor.predict(X_train, batch_size=1024, verbose=0).astype(np.float32)
GAP_val   = gap_extractor.predict(X_val,   batch_size=1024, verbose=0).astype(np.float32)
GAP_test  = gap_extractor.predict(X_test,  batch_size=1024, verbose=0).astype(np.float32)

print(f'GAP features: train={GAP_train.shape}  val={GAP_val.shape}  test={GAP_test.shape}')

gap_path = data_dir / 'cnn4feat_gap_features.h5'
with h5py.File(str(gap_path), 'w') as hf:
    hf.create_dataset('train/gap',  data=GAP_train,  compression='gzip', compression_opts=4)
    hf.create_dataset('train/y',    data=y_train)
    hf.create_dataset('train/snr',  data=snr_train)
    hf.create_dataset('val/gap',    data=GAP_val,    compression='gzip', compression_opts=4)
    hf.create_dataset('val/y',      data=y_val)
    hf.create_dataset('val/snr',    data=snr_val)
    hf.create_dataset('test/gap',   data=GAP_test,   compression='gzip', compression_opts=4)
    hf.create_dataset('test/y',     data=y_test)
    hf.create_dataset('test/snr',   data=snr_test)
    hf.attrs['source_model'] = 'model_cnn_4feat.keras'
    hf.attrs['gap_dim'] = GAP_train.shape[1]

print(f'GAP features salvas → {gap_path}')

# Métricas finais
results = {
    'auc_val':  float(auc_val),
    'auc_test': float(auc_test),
    'params':   model.count_params(),
    'gap_dim':  int(GAP_train.shape[1]),
    'alpha':    ALPHA,
    'pd_vs_snr': {str(s): (None if np.isnan(pd_cnn4feat.get(s, np.nan)) else float(pd_cnn4feat[s]))
                  for s in SNR_BINS},
    'scaler': {'mean': sc_mean.tolist(), 'std': sc_std.tolist()},
    'model_path': model_path,
}
out_path = data_dir / 'nn14_cnn_4feat_results.json'
with open(str(out_path), 'w') as fp:
    json.dump(results, fp, indent=2)

print(f'Resultados salvos → {out_path}')
print(f'Modelo salvo      → {model_path}')
print(f'\nPD vs SNR (CNN 4-feat, α={ALPHA:.0e}):')
for s in SNR_BINS:
    v = pd_cnn4feat.get(s, np.nan)
    print(f'  {s:3d} dB  PD={v:.4f}' if not np.isnan(v) else f'  {s:3d} dB  PD=N/A')

## Resumo do Ablation — Arquitetura CNN em 4 Features

| Modelo | Input shape | Params | Inductive bias |
|--------|-------------|--------|----------------|
| DNN 4-feat (NN_02b) | (4,) | ~44k | Nenhum (fully connected) |
| **CNN 4-feat (este)** | (4, 1) | ~1k | Conv local k=2, compartilhamento |
| 1D CNN z_eq (NN_02) | (1024, 1) | ~69k | Conv local k=5–15, hierárquico |

**Interpretação**:
- Se CNN-4feat ≈ DNN-4feat: o inductive bias conv não ajuda em sequências de 4 elementos.
  Os receptive fields k=2 cobrem apenas pares de features adjacentes — limitado.
- Se CNN-4feat < DNN-4feat: a arquitetura conv prejudica com input tão curto
  (o DNN tem muito mais parâmetros para explorar todas as combinações de 4 features).
- Em ambos os casos, a conclusão para o paper é que a CNN agrega valor principalmente
  quando processa o sinal bruto z_eq (1024 amostras), onde o compartilhamento de pesos
  é essencial para aprender filtros eficientes.

**Próximo**: NN_11 compara CNN-SVM com features de 128d (z_eq CNN) vs 32d (4-feat CNN).